# Abdomen CT — YOLO 2D Detection Pipeline
### Kaggle Notebook (Birincil) · Google Colab (İkincil)

**Kaggle'da çalıştırma:**
1. Sağ panelden Dataset ekle: `ramazan2020/abdomen-acikveri`
2. `Settings → Accelerator → GPU (P100/T4)` seç
3. Hücreleri sırayla çalıştır — PNG kesitleri dataset içinde **hazır gelir**

**Colab'da çalıştırma:**
1. `Runtime → Change runtime type → GPU`
2. Kaggle Secrets'a `KAGGLE_USERNAME` + `KAGGLE_KEY` ekle

---

| Adım | Süre |
|------|------|
| Kurulum + Konfigürasyon | ~3 dk |
| Eğitim (100 epoch, T4) | ~3–5 saat |
| Inference + F1 | ~15 dk |

> **Not:** Eğitim session kesilirse `YOLO_CONTINUE = True` yapıp devam edin.

---
## 0. Ortam Tespiti ve GPU Kontrolü

In [2]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

env_name = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'
print(f"Ortam : {env_name}")

# import torch
# if not torch.cuda.is_available():
#     if IS_LOCAL and torch.backends.mps.is_available():
#         print("GPU : Apple Silicon MPS (ultralytics MPS destekler ✓)")
#     else:
#         raise RuntimeError(
#             "GPU bulunamadı!\n"
#             "Kaggle: Settings → Accelerator → GPU\n"
#             "Colab : Runtime → Change runtime type → GPU"
#         )
# else:
#     print(f"GPU   : {torch.cuda.get_device_name(0)}")
#     print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
#     print(f"CUDA  : {torch.version.cuda}")

Ortam : Local


---
## 1. Ortam Kurulumu (Colab için)

In [3]:
if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print("Kaggle kimlik bilgileri Colab Secrets'tan yüklendi")
    except Exception:
        from google.colab import files
        import json as _j, shutil as _sh
        print("kaggle.json dosyasını seçin:")
        _up = files.upload()
        _kp = Path.home() / '.kaggle' / 'kaggle.json'
        _kp.parent.mkdir(parents=True, exist_ok=True)
        for fname in _up:
            _sh.move(fname, str(_kp))
        os.chmod(str(_kp), 0o600)
        _kd = _j.loads(_kp.read_text())
        os.environ['KAGGLE_USERNAME'] = _kd['username']
        os.environ['KAGGLE_KEY']      = _kd['key']
        print("kaggle.json yüklendi")

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("Google Drive bağlandı")
else:
    print("Kaggle ortamı — kurulum atlandı")

Kaggle ortamı — kurulum atlandı


---
## 2. Kütüphane Kurulumu

In [4]:
print("Kütüphaneler kuruluyor...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'ultralytics', 'pillow', 'pandas', 'numpy', 'tqdm'],
    check=True
)
import importlib; importlib.invalidate_caches()

import ultralytics
print(f"ultralytics : {ultralytics.__version__}")

import torch
print(f"PyTorch     : {torch.__version__}")

Kütüphaneler kuruluyor...
ultralytics : 8.3.240
PyTorch     : 1.12.1+cpu


---
## 3. Konfigürasyon

**Kaggle**: `KAGGLE_DATASET_SLUG` = dataset klasör adı (`/kaggle/input/` altında).  
**Colab**: Dataset Kaggle'dan indirilir.

In [5]:
import os, sys, json, shutil, time
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import List

# ─── Kullanıcı Ayarları ────────────────────────────────────────────────────
KAGGLE_DATASET_SLUG = 'abdomen-acikveri'
KAGGLE_DATASET_ID   = 'ramazan2020/abdomen-acikveri'
GITHUB_URL          = 'https://github.com/ramazan2020/abdomen1.git'
FOLD = 0

@dataclass
class YoloConfig:
    model: str = 'yolo12m.pt'
    img_size: int = 512
    batch_size: int = 16
    epochs: int = 100
    patience: int = 30
    lr0: float = 0.001
    lrf: float = 0.01
    weight_decay: float = 0.0005
    mosaic: float = 0.0
    mixup: float = 0.0
    fliplr: float = 0.0
    flipud: float = 0.0
    hsv_h: float = 0.0
    hsv_s: float = 0.0
    hsv_v: float = 0.4
    degrees: float = 10.0
    erasing: float = 0.1
    copy_paste: float = 0.7

YOLO_CFG = YoloConfig()
# ──────────────────────────────────────────────────────────────────────────

SUPER_CLASSES: List[str] = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

if IS_KAGGLE:
    DATA_DIR      = Path('/kaggle/input') / KAGGLE_DATASET_SLUG
    WORK_DIR      = Path('/kaggle/working')
    DRIVE_BASE    = None

elif IS_COLAB:
    DATA_DIR      = Path('/content/kaggle_data')
    WORK_DIR      = Path('/content')
    DRIVE_BASE    = Path('/content/drive/MyDrive/Abdomen')
    if DRIVE_BASE:
        DRIVE_BASE.mkdir(parents=True, exist_ok=True)

elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError:
        pass
    PROJECT_ROOT = Path(__file__).resolve().parent if '__file__' in dir() else Path('.').resolve()
    DATA_DIR     = PROJECT_ROOT
    WORK_DIR     = PROJECT_ROOT / 'outputs'
    DRIVE_BASE   = None

YOLO_RUNS_DIR = WORK_DIR / 'runs' / 'det'
DATASET_YAML  = WORK_DIR / 'dataset.yaml'
OUTPUT_DIR    = WORK_DIR / 'output'

DET_DATA_DIR   = DATA_DIR / 'outputs' / 'det_data' if IS_LOCAL else DATA_DIR / 'det_data'
YOLO_FOLD_DIR  = DET_DATA_DIR / f'fold{FOLD}'
YOLO_TRAIN_IMG = YOLO_FOLD_DIR / 'images' / 'train'
YOLO_VAL_IMG   = YOLO_FOLD_DIR / 'images' / 'val'
YOLO_VAL_LBL   = YOLO_FOLD_DIR / 'labels' / 'val'

YOLO_RUNS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault('ABDOMEN_OUT_DIR', str(WORK_DIR / 'abdomen_src_out'))

print(f'Ortam         : {env_name}')
print(f'DATA_DIR      : {DATA_DIR}  (var={DATA_DIR.exists()})')
print(f'YOLO_FOLD_DIR : {YOLO_FOLD_DIR}  (var={YOLO_FOLD_DIR.exists()})')
print(f'Model         : {YOLO_CFG.model}')

if not DATA_DIR.exists():
    raise FileNotFoundError(f'DATA_DIR bulunamadı: {DATA_DIR}')

Ortam         : Local
DATA_DIR      : D:\makale-pdf\Proje\code  (var=True)
YOLO_FOLD_DIR : D:\makale-pdf\Proje\code\outputs\det_data\fold0  (var=False)
Model         : yolo12m.pt


In [6]:
# ── GitHub Repo / Local src ───────────────────────────────────────────────
if IS_LOCAL:
    PROJECT_ROOT = Path(__file__).resolve().parent if '__file__' in dir() else Path('.').resolve()
    REPO_DIR = PROJECT_ROOT
    print(f'Local: src/ kullanılıyor → {REPO_DIR / "src"}')
else:
    REPO_DIR = WORK_DIR / 'abdomen1'
    if not (REPO_DIR / '.git').exists():
        print(f'Klonlanıyor: {GITHUB_URL}')
        subprocess.run(
            ['git', 'clone', '--depth=1', GITHUB_URL, str(REPO_DIR)],
            check=True
        )
    else:
        print('Repo mevcut, güncelleniyor...')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       capture_output=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.evaluation import f1_at_iou, top5_f1_mean

print(f'Repo            : {REPO_DIR}')
print('src.evaluation  ✓  (f1_at_iou, top5_f1_mean)')

Local: src/ kullanılıyor → D:\makale-pdf\Proje\code\src
Repo            : D:\makale-pdf\Proje\code
src.evaluation  ✓  (f1_at_iou, top5_f1_mean)


---
## 4. Veri Hazırlık


In [9]:
# ── Veri Hazırlık Kontrolü ─────────────────────────────────────────────────
# manifest.csv + splits + YOLO det_data, Faz1_2_VeriHazirlik notebook'u
# tarafından üretilir. Mevcut değilse burada otomatik oluşturulur.
#
# LOCAL notu: det_data yoksa bu hücre oluşturur (~30-90 dk).
# Kaggle/Colab: dataset içinde hazır gelir; bu hücre sadece doğrular.

# SPLIT_DIR YOLO cell_config'te tanımsız — burada çözümleniyor
_SPLIT_DIR = (WORK_DIR / 'splits') if IS_LOCAL else (DATA_DIR / 'outputs' / 'splits')
_SPLIT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('ABDOMEN_SPLIT_DIR', str(_SPLIT_DIR))

_manifest_csv = _SPLIT_DIR / 'manifest.csv'
_splits_csv   = _SPLIT_DIR / 'splits.csv'

if not _manifest_csv.exists():
    print('manifest.csv bulunamadı — oluşturuluyor...')
    _bilgi = DATA_DIR / 'Bilgi.xlsx'
    if not _bilgi.exists():
        raise FileNotFoundError(
            f'Bilgi.xlsx bulunamadı: {_bilgi}\n'
            "Faz1_2_VeriHazirlik notebook'unu önce çalıştırın."
        )
    os.environ.setdefault('ABDOMEN_BILGI_XLSX', str(_bilgi))
    from src.preprocessing import build_manifest
    build_manifest(_manifest_csv)
    print(f'manifest.csv oluşturuldu ✓  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')
else:
    print(f'manifest.csv mevcut ✓  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')

if not _splits_csv.exists():
    print('splits.csv bulunamadı — oluşturuluyor...')
    from src.splits import make_splits
    make_splits(out_dir=_SPLIT_DIR)
    print('splits.csv oluşturuldu ✓')
else:
    print('splits.csv mevcut ✓')

# YOLO det_data (PNG) kontrolü
if not YOLO_TRAIN_IMG.exists() or not any(YOLO_TRAIN_IMG.glob('*.png')):
    print(f'det_data bulunamadı — oluşturuluyor (~30-90 dakika)...')
    os.environ.setdefault('ABDOMEN_DET_DATA_DIR', str(DET_DATA_DIR))
    from src.detection import export_yolo_dataset
    export_yolo_dataset(fold=FOLD, include_val_negatives=False, bbox_only=True)
    _n_png = len(list(YOLO_TRAIN_IMG.glob('*.png')))
    print(f'YOLO det_data oluşturuldu ✓  ({_n_png:,} train PNG)')
else:
    _n_png = len(list(YOLO_TRAIN_IMG.glob('*.png')))
    print(f'det_data mevcut ✓  ({_n_png:,} train PNG)')

manifest.csv mevcut ✓  (4180 KB)
splits.csv mevcut ✓
det_data bulunamadı — oluşturuluyor (~30-90 dakika)...
BBox filtresi: 7,303 bbox'sız satır dışlandı → 31,965 kesit işlenecek


YOLO fold0:   8%|▊         | 2239/26434 [00:14<03:18, 122.12it/s]

---
## 4. Veri İndirme (Yalnızca Colab)

**Kaggle'da bu hücre otomatik atlanır** — PNG'ler dataset içinde hazır.

In [ ]:
if IS_COLAB:
    _already = YOLO_FOLD_DIR.exists() and YOLO_TRAIN_IMG.exists()
    if _already:
        print(f"Veri zaten mevcut: {YOLO_FOLD_DIR}")
    else:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Dataset indiriliyor: {KAGGLE_DATASET_ID}")
        print("Bu işlem 30-90 dakika sürebilir...")
        t0 = time.time()
        r = subprocess.run(
            ['kaggle', 'datasets', 'download',
             '-d', KAGGLE_DATASET_ID,
             '-p', str(DATA_DIR), '--unzip'],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print('HATA:', r.stderr[-1000:])
            raise RuntimeError('İndirme başarısız')
        print(f"İndirildi ({time.time()-t0:.0f}s)")

assert YOLO_FOLD_DIR.exists(), f'YOLO fold dizini bulunamadı: {YOLO_FOLD_DIR}'

n_train = len(list(YOLO_TRAIN_IMG.glob('*.png'))) if YOLO_TRAIN_IMG.exists() else 0
n_val   = len(list(YOLO_VAL_IMG.glob('*.png')))   if YOLO_VAL_IMG.exists()   else 0
print(f'train PNG : {n_train:,}')
print(f'val   PNG : {n_val:,}')

if n_train == 0:
    raise FileNotFoundError(
        f'PNG bulunamadı: {YOLO_TRAIN_IMG}\n'
        'Dataset içinde det_data/ klasörü eksik olabilir.'
    )

AssertionError: YOLO fold dizini bulunamadı: D:\makale-pdf\Proje\code\outputs\det_data\fold0

---
## 5. dataset.yaml Oluştur

In [ ]:
yaml_content = (
    f'path: {YOLO_FOLD_DIR}\n'
    'train: images/train\n'
    'val: images/val\n'
    'names:\n' +
    ''.join(f'  {i}: {n}\n' for i, n in enumerate(SUPER_CLASSES))
)
DATASET_YAML.write_text(yaml_content)

print(f'dataset.yaml: {DATASET_YAML}')
print(yaml_content)

---
## 6. Eğitim

Checkpoint otomatik `runs/det/` altına kaydedilir.  
Session kesilirse: `YOLO_CONTINUE = True` yapın.

In [ ]:
from ultralytics import YOLO
import torch

YOLO_CONTINUE = False  # True → son checkpoint'ten devam

# assert torch.cuda.is_available(), 'GPU gerekli!'

_run_name = f'fold{FOLD}_{YOLO_CFG.model.replace(".pt","")}'
print(f'Eğitim : {_run_name}')
# print(f'GPU    : {torch.cuda.get_device_name(0)}')
# print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Runs   : {YOLO_RUNS_DIR}')

if torch.backends.mps.is_available():
    _device, _workers = 'mps', 2
elif torch.cuda.is_available():
    _device, _workers = '0', 8
else:
    _device, _workers = 'cpu', 4

yolo_model = YOLO(YOLO_CFG.model)
yolo_model.train(
    data=str(DATASET_YAML),
    imgsz=YOLO_CFG.img_size,
    epochs=YOLO_CFG.epochs,
    batch=YOLO_CFG.batch_size,
    patience=YOLO_CFG.patience,
    lr0=YOLO_CFG.lr0,
    lrf=YOLO_CFG.lrf,
    weight_decay=YOLO_CFG.weight_decay,
    mosaic=YOLO_CFG.mosaic,
    mixup=YOLO_CFG.mixup,
    fliplr=YOLO_CFG.fliplr,
    flipud=YOLO_CFG.flipud,
    hsv_h=YOLO_CFG.hsv_h,
    hsv_s=YOLO_CFG.hsv_s,
    hsv_v=YOLO_CFG.hsv_v,
    degrees=YOLO_CFG.degrees,
    erasing=YOLO_CFG.erasing,
    copy_paste=YOLO_CFG.copy_paste,
    project=str(YOLO_RUNS_DIR),
    name=_run_name,
    device=_device,
    workers=_workers,
    seed=42,
    deterministic=False,
    resume=YOLO_CONTINUE,
    amp=True,
)

YOLO_BEST = Path(yolo_model.trainer.save_dir) / 'weights' / 'best.pt'
print(f'\nEn iyi ağırlık: {YOLO_BEST}')

if IS_COLAB and DRIVE_BASE:
    _yd = DRIVE_BASE / 'yolo_weights'
    _yd.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(YOLO_BEST), str(_yd / f'fold{FOLD}_best.pt'))
    print(f"Drive'a yedeklendi: {_yd}")

---
## 6b. Per-Class mAP (YOLO Native Val)


In [ ]:
from ultralytics import YOLO
import pandas as pd

# En iyi checkpoint'i yükle
_best = Path(yolo_model.trainer.save_dir) / 'weights' / 'best.pt' if hasattr(yolo_model, 'trainer') else YOLO_BEST
val_model = YOLO(str(_best))

# Validation
val_results = val_model.val(data=str(DATASET_YAML), split='val', verbose=False)

# Per-class mAP çıkar
_names = val_results.names          # {0: 'acute_cholecystitis', ...}
_ap50  = val_results.box.ap50       # per-class mAP@0.5
_maps  = val_results.box.maps       # per-class mAP@0.5:0.95

rows = []
for idx, name in _names.items():
    rows.append({
        'Class'       : name,
        'mAP@0.5'     : round(float(_ap50[idx]), 4),
        'mAP@0.5:0.95': round(float(_maps[idx]), 4),
    })
df_map = pd.DataFrame(rows)
df_map.loc[len(df_map)] = {
    'Class': 'ALL (mean)',
    'mAP@0.5': round(float(val_results.box.map50), 4),
    'mAP@0.5:0.95': round(float(val_results.box.map), 4),
}

print(f'=== YOLO Per-Class mAP — Fold {FOLD} ===')
print(df_map.to_string(index=False))

# Kaydet
df_map.to_csv(OUTPUT_DIR / f'yolo_fold{FOLD}_map.csv', index=False)
print(f'\nKaydedildi: {OUTPUT_DIR}/yolo_fold{FOLD}_map.csv')


---
## 7. Tahmin (Inference) — Validasyon Seti

In [ ]:
from ultralytics import YOLO
from PIL import Image
from tqdm.notebook import tqdm

# Eğitim sonrası otomatik atanır.
# Manuel yüklemek için:
# YOLO_BEST = Path('/kaggle/working/runs/det/fold0_yolo12m/weights/best.pt')
assert 'YOLO_BEST' in dir() and YOLO_BEST.exists(), \
    f'Model bulunamadı: {YOLO_BEST}'

def _parse_stem(stem: str):
    """T_20001_100007 veya C_20001_100063 → (case_int, image_id_int)"""
    parts = stem.split('_')
    if len(parts) >= 3:
        return int(parts[1]), int(parts[2])
    return int(parts[0]), int(parts[1])

_model = YOLO(str(YOLO_BEST))
_val_imgs = sorted(YOLO_VAL_IMG.glob('*.png'))
print(f'Val inference: {len(_val_imgs)} PNG')
print(f'Model        : {YOLO_BEST}')

CONF_TH = 0.05
yolo_preds = []
for ip in tqdm(_val_imgs, desc='YOLO inference'):
    case, image_id = _parse_stem(ip.stem)
    res = _model.predict(str(ip), conf=CONF_TH, verbose=False,
                          imgsz=YOLO_CFG.img_size)[0]
    for box, sc, cl in zip(res.boxes.xyxy.cpu().numpy(),
                            res.boxes.conf.cpu().numpy(),
                            res.boxes.cls.cpu().numpy()):
        yolo_preds.append({
            'case': case, 'image_id': image_id, 'class': int(cl),
            'score': float(sc),
            'x1': float(box[0]), 'y1': float(box[1]),
            'x2': float(box[2]), 'y2': float(box[3]),
        })

yolo_pred_df = pd.DataFrame(yolo_preds)
print(f'Tahmin: {len(yolo_pred_df):,} bbox  |  {yolo_pred_df["case"].nunique() if not yolo_pred_df.empty else 0} vaka')

---
## 8. Ground Truth Yükleme

In [ ]:
from PIL import Image

gt_rows = []
for lp in sorted(YOLO_VAL_LBL.glob('*.txt')):
    ip = YOLO_VAL_IMG / (lp.stem + '.png')
    if not ip.exists():
        continue
    W, H = Image.open(ip).size
    case, image_id = _parse_stem(lp.stem)
    for line in lp.read_text().strip().splitlines():
        p = line.split()
        if len(p) < 5:
            continue
        cl = int(p[0]); cx, cy, w, h = map(float, p[1:5])
        gt_rows.append({
            'case': case, 'image_id': image_id, 'class': cl,
            'x1': (cx - w/2)*W, 'y1': (cy - h/2)*H,
            'x2': (cx + w/2)*W, 'y2': (cy + h/2)*H,
        })

gt_df = pd.DataFrame(gt_rows)
print(f'GT: {len(gt_df):,} bbox  |  {gt_df["case"].nunique() if not gt_df.empty else 0} vaka')
if not gt_df.empty:
    print(gt_df.groupby('class').size()
            .rename(index={i: SUPER_CLASSES[i] for i in range(len(SUPER_CLASSES))}))

---
## 9. Değerlendirme — F1 Skoru

In [ ]:
# f1_at_iou ve top5_f1_mean src.evaluation'dan import edildi (cell_github)
if yolo_pred_df.empty:
    print('Prediction boş — inference adımını kontrol edin.')
else:
    top5   = top5_f1_mean(yolo_pred_df, gt_df)
    detail = f1_at_iou(yolo_pred_df, gt_df, iou_th=0.3)

    print(f'=== YOLO — Fold {FOLD} Sonuçları ===')
    print(f'Top-5 Mean F1      : {top5["top5_mean_f1"]:.4f}')
    print(f'Macro F1 @ IoU=0.3 : {detail["macro_f1"]:.4f}')
    print(f'Micro F1 @ IoU=0.3 : {detail["micro_f1"]:.4f}')
    print()
    print('IoU eşiğine göre Macro F1:')
    for th, f1v in top5['per_threshold']:
        print(f'  {th:.1f}: {f1v:.4f}')
    print()
    print('Sınıf bazında @ IoU=0.3:')
    for cls_id, cls_name in enumerate(SUPER_CLASSES):
        if cls_id in detail['per_class']:
            m = detail['per_class'][cls_id]
            print(f'  {cls_id} {cls_name:<30}  '
                  f'P={m["precision"]:.3f} R={m["recall"]:.3f} F1={m["f1"]:.3f}  '
                  f'TP={m["tp"]} FP={m["fp"]} FN={m["fn"]}')

---
## 10. Görsel Kontrol

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
%matplotlib inline

N_SAMPLES  = 4
CONF_SHOW  = 0.25  # Görselde min conf eşiği

random.seed(42)
_val_imgs_all = sorted(YOLO_VAL_IMG.glob('*.png'))

# Annotasyonlu görüntüleri tercih et
_annotated = [ip for ip in _val_imgs_all
              if (YOLO_VAL_LBL / (ip.stem + '.txt')).exists()
              and (YOLO_VAL_LBL / (ip.stem + '.txt')).stat().st_size > 0]
_samples = random.sample(_annotated, min(N_SAMPLES, len(_annotated)))

fig, axes = plt.subplots(1, len(_samples), figsize=(5*len(_samples), 5))
if len(_samples) == 1: axes = [axes]

for ax, ip in zip(axes, _samples):
    img = np.array(Image.open(ip).convert('RGB'))
    ax.imshow(img)
    case, image_id = _parse_stem(ip.stem)

    # GT (mavi kesikli)
    for _, r in gt_df[(gt_df['case']==case) & (gt_df['image_id']==image_id)].iterrows():
        ax.add_patch(mpatches.Rectangle(
            (r.x1, r.y1), r.x2-r.x1, r.y2-r.y1,
            linewidth=2, edgecolor='cyan', facecolor='none', linestyle='--'
        ))
        ax.text(r.x1, r.y1-4, SUPER_CLASSES[int(r['class'])][:12],
                color='cyan', fontsize=7, backgroundcolor='black')

    # Tahmin (kırmızı)
    _pr = yolo_pred_df[(yolo_pred_df['case']==case) &
                        (yolo_pred_df['image_id']==image_id) &
                        (yolo_pred_df['score']>=CONF_SHOW)]
    for _, r in _pr.iterrows():
        ax.add_patch(mpatches.Rectangle(
            (r.x1, r.y1), r.x2-r.x1, r.y2-r.y1,
            linewidth=2, edgecolor='red', facecolor='none'
        ))
        ax.text(r.x1, r.y2+10, f"{SUPER_CLASSES[int(r['class'])][:10]} {r.score:.2f}",
                color='red', fontsize=7, backgroundcolor='white')

    ax.set_title(f'case={case} slice={image_id}', fontsize=8)
    ax.axis('off')

plt.suptitle(f'Cyan=GT  Red=Pred(conf≥{CONF_SHOW})  Fold {FOLD}', fontsize=11)
plt.tight_layout()
plt.show()

---
## 11. Sonuçları Kaydet

**Kaggle**: `/kaggle/working/output/` altına kaydedilir; `Save Version` ile output olur.  
**Colab**: Drive'a kopyalanır.

In [ ]:
if yolo_pred_df.empty:
    print('Prediction boş, kaydetme atlandı.')
else:
    yolo_pred_df.to_csv(OUTPUT_DIR / f'yolo_fold{FOLD}_pred.csv', index=False)

    summary = {
        'fold'          : FOLD,
        'model'         : YOLO_CFG.model,
        'top5_mean_f1'  : top5['top5_mean_f1'],
        'macro_f1_03'   : detail['macro_f1'],
        'micro_f1_03'   : detail['micro_f1'],
        'per_threshold' : {str(th): float(f1v) for th, f1v in top5['per_threshold']},
        'per_class_03'  : {
            SUPER_CLASSES[cid]: {
                k: round(v, 4) if isinstance(v, float) else v
                for k, v in m.items()
            }
            for cid, m in detail['per_class'].items()
        },
    }
    (OUTPUT_DIR / f'yolo_fold{FOLD}_summary.json').write_text(
        json.dumps(summary, indent=2, ensure_ascii=False)
    )

    print(f'Çıktılar: {OUTPUT_DIR}')
    for f in sorted(OUTPUT_DIR.glob('*')):
        print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

    print()
    print('=' * 48)
    print(f'  YOLO Fold {FOLD} — Özet')
    print('=' * 48)
    print(f'  Top-5 Mean F1      : {top5["top5_mean_f1"]:.4f}')
    print(f'  Macro F1 @ IoU=0.3 : {detail["macro_f1"]:.4f}')
    print(f'  Micro F1 @ IoU=0.3 : {detail["micro_f1"]:.4f}')
    print('=' * 48)

    if IS_COLAB and DRIVE_BASE:
        _dst = DRIVE_BASE / 'output'
        _dst.mkdir(parents=True, exist_ok=True)
        for f in OUTPUT_DIR.glob('yolo_*'):
            shutil.copy2(str(f), str(_dst / f.name))
        print(f"Drive'a kopyalandı: {_dst}")

    if IS_KAGGLE:
        print('\nKaggle çıktıları kaydetmek için:')
        print('  Save Version → sağ üst → Save & Run All')